In [1]:
# Importing relevant packages
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

In [2]:
# Importing gym csv
workouts = pd.read_csv('data/gym.csv')

In [3]:
# Formatting column headings
workouts.columns = workouts.columns.str.lower().str.replace(' ', '_')

In [4]:
# Drop irrelevant cols
workouts.drop(
    columns=[
        'description',
        'superset_id',
        'exercise_notes',
        'set_type',
        'distance_km',
        'duration_seconds',
        'rpe'
    ],
    inplace=True
)

In [5]:
# Renaming columns
workouts.rename(columns={
    'title':'category',
    'set_index':'set_num',
    'exercise_title':'name'
}, inplace=True)

In [6]:
# Setting datetimes for cols
workouts['start_time'] = pd.to_datetime(
    workouts['start_time'],
    format='%b %d, %Y, %I:%M %p'
)

workouts['end_time'] = pd.to_datetime(
    workouts['end_time'],
    format='%b %d, %Y, %I:%M %p'
)

In [7]:
# Only including workouts from 2026 and above
workouts = workouts[workouts['start_time'].dt.year > 2025]

In [8]:
# Dropping incorrect workout rows
workouts.drop(workouts[workouts['name'] == 'Glute Ham Raise'].index, inplace=True)

In [9]:
# Assigning missing weight_kg to correct values
dates = ['2026-05-07', '2026-05-14', '2026-05-20']

date_mask = workouts['start_time'].dt.date.isin([pd.Timestamp(d).date() for d in dates])
name_mask = workouts['name'] == 'Standing Calf Raise'

workouts.loc[date_mask & name_mask, 'weight_kg'] = 8.0

In [10]:
# Assigning missing weight_kg to correct values
date_mask = workouts['start_time'].dt.date == pd.Timestamp('2026-05-27').date()
workouts.loc[date_mask & name_mask, 'weight_kg'] = 10.0

In [11]:
# Fixing set_nums
workouts['set_num'] = workouts['set_num'] + 1

In [12]:
# Assign unique workout_id based on start_time & end_time
workouts['workout_id'] = workouts.groupby(['start_time', 'end_time']).ngroup() + 1

# Assign unique exercise_id based on name
workouts['exercise_id'] = workouts.groupby('name').ngroup() + 1

In [13]:
# Rename skullcrusher to correct exercise
workouts['name'] = workouts['name'].replace('Skullcrusher (Dumbbell)', 'Triceps Rope Pushdown')

In [14]:
# Assign muscle group to each exercise name
muscle_groups = {
    "Incline Bench Press (Barbell)": "chest",
    "Bench Press (Dumbbell)": "chest",
    "Shoulder Press (Dumbbell)": "shoulders",
    "Lateral Raise (Dumbbell)": "shoulders",
    "Face Pull": "shoulders",
    "Triceps Rope Pushdown": "triceps",
    "Bent Over Row (Barbell)": "back",
    "Lat Pulldown (Cable)": "back",
    "Chest Supported Incline Row (Dumbbell)": "back",
    "Bicep Curl (Barbell)": "biceps",
    "Squat (Barbell)": "quads",
    "Bulgarian Split Squat": "quads",
    "Standing Calf Raise": "calves",
    "Seated Calf Raise": "calves"
}

workouts['muscle_group'] = workouts['name'].map(muscle_groups)

In [15]:
workouts.sort_values('start_time')

,category,start_time,end_time,name,set_num,weight_kg,reps,workout_id,exercise_id,muscle_group
329,PUSH,2026-05-04 13:59:00,2026-05-04 14:40:00,Triceps Rope Pushdown,3,7.5,10,1,12,triceps
315,PUSH,2026-05-04 13:59:00,2026-05-04 14:40:00,Incline Bench Press (Barbell),1,20.0,12,1,7,chest
316,PUSH,2026-05-04 13:59:00,2026-05-04 14:40:00,Incline Bench Press (Barbell),2,25.0,12,1,7,chest
317,PUSH,2026-05-04 13:59:00,2026-05-04 14:40:00,Incline Bench Press (Barbell),3,30.0,12,1,7,chest
318,PUSH,2026-05-04 13:59:00,2026-05-04 14:40:00,Shoulder Press (Dumbbell),1,10.0,12,1,11,shoulders
...,...,...,...,...,...,...,...,...,...,...
12,PULL,2026-06-09 13:06:00,2026-06-09 13:49:00,Face Pull,1,12.5,12,23,6,shoulders
13,PULL,2026-06-09 13:06:00,2026-06-09 13:49:00,Face Pull,2,12.5,12,23,6,shoulders
14,PULL,2026-06-09 13:06:00,2026-06-09 13:49:00,Face Pull,3,12.5,12,23,6,shoulders
7,PULL,2026-06-09 13:06:00,2026-06-09 13:49:00,Chest Supported Incline Row (Dumbbell),2,10.0,11,23,5,back


In [16]:
# Create workouts table
df_workouts = workouts[['workout_id', 'start_time', 'end_time', 'category']].drop_duplicates().sort_values('workout_id')

In [17]:
# Create exercises table
df_exercises = workouts[['exercise_id', 'name', 'muscle_group']].drop_duplicates().sort_values('exercise_id')

In [18]:
# Create workout_exercises table
df_workout_exercises = workouts[['workout_id', 'exercise_id', 'set_num', 'weight_kg', 'reps']].sort_values('workout_id')

In [19]:
load_dotenv()
engine = create_engine(os.getenv('DATABASE_URL'))

In [20]:
def upsert_data(df_workouts, df_exercises, df_workout_exercises):
    
    # workouts — skip any workout_id already in DB
    existing_workouts = pd.read_sql('SELECT workout_id FROM workouts', con=engine)
    new_workouts = df_workouts[~df_workouts['workout_id'].isin(existing_workouts['workout_id'])]
    if not new_workouts.empty:
        new_workouts.to_sql('workouts', con=engine, if_exists='append', index=False)
        print(f'Added {len(new_workouts)} new workouts')
    else:
        print('No new workouts')

    # exercises — skip any exercise already in DB
    existing_exercises = pd.read_sql('SELECT name FROM exercises', con=engine)
    new_exercises = df_exercises[~df_exercises['name'].isin(existing_exercises['name'])]
    if not new_exercises.empty:
        new_exercises.to_sql('exercises', con=engine, if_exists='append', index=False)
        print(f'Added {len(new_exercises)} new exercises')
    else:
        print('No new exercises')

    # workout_exercises — skip any workout_id already in DB
    existing_we = pd.read_sql('SELECT DISTINCT workout_id FROM workout_exercises', con=engine)
    new_we = df_workout_exercises[~df_workout_exercises['workout_id'].isin(existing_we['workout_id'])]
    if not new_we.empty:
        new_we.to_sql('workout_exercises', con=engine, if_exists='append', index=False)
        print(f'Added {len(new_we)} new sets')
    else:
        print('No new sets')

upsert_data(df_workouts, df_exercises, df_workout_exercises)

No new workouts
No new exercises
No new sets
